# 03 — EfficientNet-B0 transfer learning + Grad-CAM

**Disconnect-proof:** the best-val-F1 weights are saved to Drive
(`MyDrive/melanoma/checkpoints/effnet_b0_best.pt`) the moment val F1
improves. If the runtime dies mid-training, just re-run the recovery
cell at the bottom — no retraining needed.

Two-stage training:
1. Freeze backbone, train classifier head only — lr=1e-3, 5 epochs.
2. Unfreeze last **2** EfficientNet blocks — lr=1e-4, up to 15 epochs,
   early stopping on val F1 (patience=5).

Class imbalance handled by **softened (sqrt-balanced)** weights —
full `balanced` weights overcorrect and tank precision.

In [ ]:
# --- Colab setup: ensure the project is on sys.path, mount Drive, load config ---
import os, sys, subprocess
from pathlib import Path

# Either the project is already on disk (uploaded zip / mounted Drive) or we
# clone it from GitHub. We never destroy local changes.
REPO_URL = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
CANDIDATE_PATHS = [
    Path.cwd(),
    Path("/content/melanoma-detection-ham10000"),
    Path("/content/drive/MyDrive/melanoma-detection-ham10000"),
]

project_root = None
for p in CANDIDATE_PATHS:
    if (p / "src").exists() and (p / "config.py").exists():
        project_root = p
        break

if project_root is None:
    project_root = Path("/content/melanoma-detection-ham10000")
    subprocess.run(["git", "clone", REPO_URL, str(project_root)], check=True)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Mount Drive (silently re-uses an existing mount on re-run)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except (ImportError, ModuleNotFoundError):
    pass

import config
config.ensure_drive_dirs()
print("Project root:", project_root)
print("Drive root  :", config.DRIVE_ROOT)
print("Data dir    :", config.DATA_DIR)
print("Results dir :", config.RESULTS_DIR)

In [ ]:
import random, numpy as np, torch
random.seed(config.SEED); np.random.seed(config.SEED); torch.manual_seed(config.SEED)
torch.cuda.manual_seed_all(config.SEED)

In [ ]:
# --- Build PyTorch Datasets / DataLoaders (data auto-cached to /content) ---
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from src.data import load_arrays, softened_class_weights

X, y, ids, idx_train, idx_val, idx_test = load_arrays(config.DATA_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "  X:", X.shape)

class HAMDataset(Dataset):
    def __init__(self, X, y, indices, transform):
        self.X, self.y, self.idx, self.tf = X, y, indices, transform
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        k = self.idx[i]
        return self.tf(self.X[k]), int(self.y[k])

# Storage is at 448x448. EfficientNet-B0 was trained at 224 in the original
# timm recipe, so we downsample explicitly here.
LEGACY_INPUT = 224
train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((LEGACY_INPUT, LEGACY_INPUT), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(0.15, 0.15, 0.15, 0.07),
    transforms.RandomResizedCrop(LEGACY_INPUT, scale=(0.85, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(config.IMAGENET_MEAN, config.IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((LEGACY_INPUT, LEGACY_INPUT), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(config.IMAGENET_MEAN, config.IMAGENET_STD),
])

bs = config.EFFNET_BATCH_SIZE
train_ld = DataLoader(HAMDataset(X, y, idx_train, train_tf), batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True)
val_ld   = DataLoader(HAMDataset(X, y, idx_val,   eval_tf),  batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
test_ld  = DataLoader(HAMDataset(X, y, idx_test,  eval_tf),  batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# --- Build model ---
from src.models import build_efficientnet_b0, freeze_backbone, unfreeze_last_n_blocks, trainable_params
model = build_efficientnet_b0(num_classes=2, pretrained=True).to(device)
freeze_backbone(model)
print("Stage-1 trainable params:", trainable_params(model))

In [ ]:
# --- Softened (sqrt-balanced) class-weighted loss ---
import torch.nn as nn
cw = softened_class_weights(y[idx_train], power=config.EFFNET_CW_POWER)
weights = torch.tensor([cw[0], cw[1]], dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=weights)
print("Softened class weights:", cw)

In [ ]:
# --- Stage 1: head only ---
import time, torch
from src.training import train_loop, EpochLog

CHECKPOINT_PATH = config.CHECKPOINT_DIR / "effnet_b0_best.pt"
print("Best weights will be persisted to:", CHECKPOINT_PATH)

opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=config.EFFNET_HEAD_LR)
log = EpochLog()
t0 = time.time()
log = train_loop(model, train_ld, val_ld, criterion, opt, device,
                 epochs=config.EFFNET_HEAD_EPOCHS, log=log,
                 checkpoint_path=CHECKPOINT_PATH)

In [ ]:
# --- Stage 2: unfreeze last N blocks ---
unfreeze_last_n_blocks(model, n=config.EFFNET_UNFREEZE_LAST_N_BLOCKS)
print(f"Stage-2 unfrozen blocks: {config.EFFNET_UNFREEZE_LAST_N_BLOCKS}")
print(f"Stage-2 trainable params: {trainable_params(model)}")

opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=config.EFFNET_FT_LR)
log = train_loop(model, train_ld, val_ld, criterion, opt, device,
                 epochs=config.EFFNET_FT_EPOCHS,
                 early_stop_patience=config.EFFNET_EARLY_STOP_PATIENCE, log=log,
                 checkpoint_path=CHECKPOINT_PATH)
train_time = time.time() - t0
print(f"Total training time: {train_time:.1f}s. Best weights at {CHECKPOINT_PATH}")

In [ ]:
# --- Plot loss / F1 curves ---
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(log.train_loss, label="train"); ax[0].plot(log.val_loss, label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(log.val_f1); ax[1].set_title("Val F1"); ax[1].set_xlabel("epoch")
fig.tight_layout(); fig.savefig(config.RESULTS_DIR / "deep_learning_effnet_curves.png", dpi=120)
plt.show()

In [ ]:
# --- Threshold optimization on val: find the F1-maximizing decision threshold ---
import numpy as np
from sklearn.metrics import f1_score
from src.training import predict

y_val_true, _, y_val_prob = predict(model, val_ld, device)
ths = np.linspace(0.05, 0.95, 181)
f1s = [f1_score(y_val_true, (y_val_prob > t).astype(int), zero_division=0) for t in ths]
best_threshold = float(ths[int(np.argmax(f1s))])
print(f"Best val threshold: {best_threshold:.3f}  ->  val F1 = {max(f1s):.4f}")

In [ ]:
# --- Evaluate on test set with the optimized threshold ---
import time
from src.evaluation import save_standard_outputs

t0 = time.time()
y_true, _, y_prob = predict(model, test_ld, device)
inf_ms = (time.time() - t0) * 1000.0 / len(y_true)
y_pred = (y_prob > best_threshold).astype(int)

hp = dict(
    arch="efficientnet_b0_timm", input_size=config.IMG_SIZE, batch_size=bs,
    head_lr=config.EFFNET_HEAD_LR, head_epochs=config.EFFNET_HEAD_EPOCHS,
    ft_lr=config.EFFNET_FT_LR,     ft_epochs=config.EFFNET_FT_EPOCHS,
    early_stop_patience=config.EFFNET_EARLY_STOP_PATIENCE,
    unfreeze_last_n_blocks=config.EFFNET_UNFREEZE_LAST_N_BLOCKS,
    class_weight=f"softened (power={config.EFFNET_CW_POWER})",
    augmentation="hflip+vflip+rot30+colorjitter+resizedcrop", hair_removal=True,
    decision_threshold=best_threshold,
    threshold_selection="argmax F1 on validation set",
)
metrics = save_standard_outputs(
    method_name="deep_learning_effnet",
    results_dir=config.RESULTS_DIR,
    y_true=y_true, y_pred=y_pred, y_prob=y_prob,
    ids=ids[idx_test],
    hyperparameters=hp,
    train_time_sec=train_time,
    inference_time_per_image_ms=inf_ms,
)
{k: v for k, v in metrics.items() if k != "hyperparameters"}

In [ ]:
# --- Grad-CAM grid: 4 melanoma + 4 non-melanoma test images ---
import numpy as np, torch, matplotlib.pyplot as plt
from src.gradcam import GradCAM, overlay_heatmap

model.eval()
cam = GradCAM(model, model.blocks[-1])

mel_idx    = idx_test[y[idx_test] == 1][:4]
nonmel_idx = idx_test[y[idx_test] == 0][:4]
chosen = list(mel_idx) + list(nonmel_idx)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, k in zip(axes.flat, chosen):
    img = X[k]
    t = eval_tf(img).unsqueeze(0).to(device)
    heat = cam(t, class_idx=int(y[k]))
    ax.imshow(overlay_heatmap(img, heat))
    ax.set_title(f"id={ids[k]}  true={int(y[k])}")
    ax.axis("off")
cam.close()
fig.tight_layout(); fig.savefig(config.RESULTS_DIR / "gradcam_grid.png", dpi=120)
plt.show()

---
## Recovery cell (use only if runtime disconnected during training)

If the cell above showing curves never ran, but you saw at least one
`-> checkpoint saved` line during training, the best weights are in
`MyDrive/melanoma/checkpoints/effnet_b0_best.pt`. Run this single cell
on a fresh runtime (after re-running the preamble + dataset cells) to
finish evaluation without retraining.

In [ ]:
# --- Recovery: load checkpoint from Drive, evaluate, save standardized outputs ---
import torch, time, numpy as np
from sklearn.metrics import f1_score
from src.training import predict
from src.evaluation import save_standard_outputs

CHECKPOINT_PATH = config.CHECKPOINT_DIR / "effnet_b0_best.pt"
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

y_val_true, _, y_val_prob = predict(model, val_ld, device)
ths = np.linspace(0.05, 0.95, 181)
best_threshold = float(ths[int(np.argmax([f1_score(y_val_true,(y_val_prob>t).astype(int), zero_division=0) for t in ths]))])

t0 = time.time()
y_true, _, y_prob = predict(model, test_ld, device)
inf_ms = (time.time() - t0) * 1000.0 / len(y_true)
y_pred = (y_prob > best_threshold).astype(int)

metrics = save_standard_outputs(
    method_name="deep_learning_effnet",
    results_dir=config.RESULTS_DIR,
    y_true=y_true, y_pred=y_pred, y_prob=y_prob,
    ids=ids[idx_test],
    hyperparameters={"recovered_from_checkpoint": True, "decision_threshold": best_threshold},
    train_time_sec=0.0,
    inference_time_per_image_ms=inf_ms,
)
{k: v for k, v in metrics.items() if k != "hyperparameters"}